In [12]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astroquery.gaia import Gaia
import tqdm
from astropy.table import join
from scipy.optimize import root_scalar
from astroquery.simbad import Simbad
from ipyaladin import Aladin
from astroquery.casda import Casda
from astropy.coordinates import SkyCoord
from astroquery.utils.tap.core import TapPlus
import astropy.units as u
import anywidget
import ipywidgets
import ipyaladin

In [13]:
OPAL_USER = 'jfer9940@uni.sydney.edu.au'
# casda = Casda()
Casda.login(username=OPAL_USER, store_password=True)

INFO: Authenticating jfer9940@uni.sydney.edu.au on CASDA ... [astroquery.casda.core]
INFO: Authentication successful! [astroquery.casda.core]


In [14]:
sources = Table.read('data/simbad_classifications.csv')

In [15]:
sources

main_id,sp_type,otype,notes
str19,str8,str3,str43
HD 175897,G5V,Ro*,--
HD 162186,G3IV(e),Ro*,--
BD+21 20A,--,*,solar type dwarf (Nascimbeni 2016)
BD+66 1664,G5,SB*,--
HD 217344,G4V,RS*,--
BD+09 4984B,F2IV/G5V,SB*,"listed as A2 (Anderson 2012, Raghavan 2010)"
V* V369 Gem,G5,RS*,--
HD 146696,G0,SB*,--
HD 156097,G5V,Er*,--


In [5]:
srsc_id_table = Table.read('data/2026.01.16_12_07_51_SRSC_Static.csv')

In [6]:
source_names = list(sources['main_id'])

In [7]:
solar_like_id_table = srsc_id_table[np.isin(srsc_id_table['Simbad_ID'], source_names)]

In [8]:
solar_like_id_table

Identifier,Simbad_ID,GaiaDR3_ID,TYCHO_ID,2MASS_ID,GCVS_ID,HIP_ID,UCAC4_ID,Archival_Survey,Archival_ID,Archival_epoch,Archival_ra,Archival_ra_err,Archival_dec,Archival_dec_err,Archival_ra_hms,Archival_dec_dms,Archival_parallax,Archival_parallax_err,Archival_pmra,Archival_pmra_err,Archival_pmdec,Archival_pmdec_err,Radio_multiple,Search_method_bin,1RXS_ID,TIC_ID,WISE_ID,AllWISE_ID,CNS5_ID,Stripped_Identifier,Stripped_Archival_ID
str10,str31,str28,str15,str23,str14,str10,str16,str39,str28,str19,float64,float64,float64,float64,str17,str19,float64,float64,float64,float64,float64,float64,float64,int64,str21,str14,str25,str25,str9,str9,str26
SRSC 00006,HD 102982,Gaia DR3 5368777294028600448,TYC 8228-2739-1,2MASS J11510909-5152323,--,--,UCAC4 191-064625,GaiaDR3,Gaia DR3 5368777294028600448,J2016,177.787504285457,1e-05,-51.876071495961,1e-05,11:51:09,-51:52:33.85738546,18.8,0.02,-57.34,0.02,-83.67,0.01,0.0,18,1RXS J115108.5-515223,TIC 94637805,--,--,--,SRSC00006,GaiaDR35368777294028600448
SRSC 00036,V* V376 Cep,Gaia DR3 2299477768899190400,TYC 4649-3690-1,2MASS J21581966+8252158,V* V376 Cep,HIP 108461,UCAC4 865-010582,GaiaDR3,Gaia DR3 2299477768899190400,J2016,329.577295376914,2e-05,82.870902021925,1e-05,21:58:19,+82:52:15.24727893,25.05,0.02,-137.33,0.02,-37.23,0.02,0.0,2,1RXS J215821.0+825218,TIC 470122267,--,--,--,SRSC00036,GaiaDR32299477768899190400
SRSC 00175,HD 165590A,--,TYC 1566-747-1,--,V* V772 Her,HIP 88637A,--,TYCHO,TYC 1566-747-1,J2000,271.457229,--,21.44599,--,18:05:49.7350,+21:26:45.56,--,--,-29.2,--,-40.3,--,2.0,2,--,--,--,--,--,SRSC00175,TYC1566-747-1
SRSC 00248,HD 82533,Gaia DR3 5312052698887995392,TYC 8584-268-1,2MASS J09304256-5331182,--,--,UCAC4 183-037492,GaiaDR3,Gaia DR3 5312052698887995392,J2016,142.67769359508,1e-05,-53.521472152447,1e-05,09:30:43,-53:31:17.29974881,18.37,0.02,43.94,0.02,52.09,0.02,0.0,18,1RXS J093042.6-533120,TIC 298079609,--,--,--,SRSC00248,GaiaDR35312052698887995392
SRSC 00255,HD 217344,Gaia DR3 6554844401276275712,TYC 7509-679-1,2MASS J23002815-3344422,V* TZ PsA,HIP 113598,UCAC4 282-218992,GaiaDR3,Gaia DR3 6554844401276275712,J2016,345.11766870906,2e-05,-33.745748964489,2e-05,23:00:28,-33:44:44.69627216,11.93,0.03,61.64,0.02,-154.0,0.02,0.0,2,1RXS J230028.2-334438,TIC 229066844,--,--,--,SRSC00255,GaiaDR36554844401276275712
SRSC 00267,HD 8558,Gaia DR3 4909846500703006976,TYC 8477-5-1,2MASS J01232126-5728507,--,HIP 6485,UCAC4 163-001271,GaiaDR3,Gaia DR3 4909846500703006976,J2016,20.839329303534,1e-05,-57.480907517274,1e-05,01:23:21,-57:28:51.26706218,22.09,0.02,92.85,0.02,-36.14,0.02,0.0,2,1RXS J012320.9-572853,TIC 206592394,--,--,--,SRSC00267,GaiaDR34909846500703006976
SRSC 00601,HD 200244,Gaia DR3 6807464138426082816,TYC 6923-323-1,2MASS J21025831-2256008,--,--,UCAC4 336-203460,GaiaDR3,Gaia DR3 6807464138426082816,J2016,315.742708658278,2e-05,-22.933914165698,1e-05,21:02:58,-22:56:02.09099651,7.17,0.02,-48.27,0.02,-62.09,0.02,0.0,2,1RXS J210258.7-225603,TIC 444149024,--,--,--,SRSC00601,GaiaDR36807464138426082816
SRSC 00628,HD 162186,Gaia DR3 4043524762244704000,TYC 7382-20-1,2MASS J17510684-3218276,--,HIP 87368,UCAC4 289-131340,GaiaDR3,Gaia DR3 4043524762244704000,J2016,267.778490987213,2e-05,-32.307805803789,1e-05,17:51:07,-32:18:28.10089364,7.43,0.02,-6.68,0.02,-27.99,0.02,0.0,2,--,TIC 444656050,--,--,--,SRSC00628,GaiaDR34043524762244704000
SRSC 00647,HD 104919,Gaia DR3 6054046850929570560,TYC 8982-3213-1,2MASS J12044888-6409555,--,--,UCAC4 130-062373,GaiaDR3,Gaia DR3 6054046850929570560,J2016,181.203204847798,8e-06,-64.165413875787,9e-06,12:04:49,-64:09:55.489952834,9.89,0.01,-38.22,0.01,-3.95,0.01,0.0,2,1RXS J120448.2-640942,TIC 379769581,--,--,--,SRSC00647,GaiaDR36054046850929570560


I downloaded a fits image for our first source from EMU. I want to see if I can view this image inside python using Aladin as well as opening it on my desktop. Eventually, we want to be able to download and view these cutouts using python alone and not directly interfacing with CASDA.

In [9]:
ra = solar_like_id_table[0]['Archival_ra']
dec = solar_like_id_table[0]['Archival_dec']
coords = SkyCoord(ra = ra*u.deg, dec = dec*u.deg, frame='icrs')

In [10]:
result_table = Casda.query_region(coords, radius="30 arcmin")
public_data = Casda.filter_out_unreleased(result_table)
mask = (public_data['dataproduct_subtype']=='cont.restored.t0') & (public_data['pol_states']=='/I/') & (public_data['quality_level']=='GOOD') & (public_data['obs_collection']=='EMU')
results = public_data[mask]

In [12]:
results

distance,dataproduct_type,calib_level,obs_collection,obs_id,obs_publisher_did,access_url,access_format,access_estsize,target_name,s_ra,s_dec,s_fov,s_region,s_resolution,s_xel1,s_xel2,t_min,t_max,t_exptime,t_resolution,t_xel,em_min,em_max,em_res_power,em_xel,o_ucd,pol_states,pol_xel,facility_name,instrument_name,dataproduct_subtype,em_ucd,em_unit,em_resolution,s_resolution_min,s_resolution_max,s_ucd,s_unit,obs_release_date,quality_level,thumbnail_id,filename
deg,,,,,,,,kbyte,,deg,deg,deg,,arcsec,,,d,d,s,s,,m,m,,,,,,,,,,,m,arcsec,arcsec,,,,,,
float64,str255,int32,str255,str255,str255,str2000,str255,int64,str255,float64,float64,float64,str200,float64,int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,int64,str255,str255,int64,str255,str255,str255,str255,str255,float64,float64,float64,str255,str255,str24,str15,int64,str1000
3.9933255102032486,cube,2,EMU,ASKAP-64419,cube-394734,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-394734,application/x-votable+xml;content=datalink,809050,,184.08913809309,-51.122331921589,63.92339537292168,POLYGON ICRS 191.49237963097 -54.650941958449 176.68560482162 -54.650717953098 177.79059120559 -47.19484175971 190.38795798846 -47.195032484822,12.282998278004857,15377,13469,60530.15661921296,60530.57341319444,36011.0,--,1,0.31774808433904084,0.31774808467581994,943491000.0,1,phot.flux.density,/I/,1,ASKAP,,cont.restored.t0,em.freq,m,3.3677910460383487e-10,10.253705962385844,12.282998278004857,pos.eq,deg,2024-08-29T00:32:44.234Z,GOOD,--,image.i.EMU_1216-51.SB64419.cont.taylor.0.restored.raw.fits
3.9933255102032486,cube,2,EMU,ASKAP-64419,cube-394735,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-394735,application/x-votable+xml;content=datalink,809050,,184.08913809309,-51.122331921589,63.92339537292168,POLYGON ICRS 191.49237963097 -54.650941958449 176.68560482162 -54.650717953098 177.79059120559 -47.19484175971 190.38795798846 -47.195032484822,7.800000000000012,15377,13469,60530.15661921296,60530.57341319444,36011.0,--,1,0.31774808433904084,0.31774808467581994,943491000.0,1,phot.flux.density,/I/,1,ASKAP,,cont.restored.t0,em.freq,m,3.3677910460383487e-10,6.999999999999984,7.800000000000012,pos.eq,deg,2024-08-29T00:32:44.234Z,GOOD,--,image.i.EMU_1216-51.SB64419.cont.taylor.0.highres.restored.conv.fits
3.9933255102032486,cube,2,EMU,ASKAP-64419,cube-394728,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-394728,application/x-votable+xml;content=datalink,809050,,184.08913809309,-51.122331921589,63.92339537292168,POLYGON ICRS 191.49237963097 -54.650941958449 176.68560482162 -54.650717953098 177.79059120559 -47.19484175971 190.38795798846 -47.195032484822,15.000000000000012,15377,13469,60530.15661921296,60530.57341319444,36011.0,--,1,0.31774808433904084,0.31774808467581994,943491000.0,1,phot.flux.density,/I/,1,ASKAP,,cont.restored.t0,em.freq,m,3.3677910460383487e-10,15.000000000000012,15.000000000000012,pos.eq,deg,2024-08-29T00:32:44.234Z,GOOD,--,image.i.EMU_1216-51.SB64419.cont.taylor.0.restored.conv.fits
4.083851237246987,cube,2,EMU,ASKAP-47138,cube-119209,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-119209,application/x-votable+xml;content=datalink,815696,,175.49753346505,-55.730114684794,64.44857433544692,POLYGON ICRS 183.90951210745013 -59.22907331294925 167.08508318457007 -59.2287358342378 168.5532757314101 -51.75457163440103 182.44224690492985 -51.75485054717199,15.000000000000012,15438,13526,59952.65041550926,59953.06720949074,36011.0,--,1,0.31774808433904084,0.31774808467581994,943491000.0,1,phot.flux.density,/I/,1,ASKAP,,cont.restored.t0,em.freq,m,3.3677910460383487e-10,15.000000000000012,15.000000000000012,pos.eq,deg,2023-02-02T08:50:14.894Z,GOOD,--,image.i.EMU_1141-55.SB47138.cont.taylor.0.restored.conv.fits
4.083851237246987,cube,2,EMU,ASKAP-47138,cube-119215,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-119215,application/x-votable+xml;content=datalink,815696,,175.49753346505,-55.730114684794,64.4

In [17]:
single_row_table = results[0:1]
url_list = Casda.cutout(single_row_table, coordinates=coords, radius=30*u.arcmin)

In [13]:
url_list

NameError: name 'url_list' is not defined

In [23]:
file_list = Casda.download_files(url_list, savedir='cutouts')

# Viewing

In [25]:
aladin = Aladin()
aladin

In [26]:
aladin.add_fits("data/image.i.EMU_1930-55.SB74517.cont.taylor.0.restored.conv.fits")

In [27]:
aladin.add_fits("cutouts/cutout-1345004-imagecube-394734.fits")

In [28]:
ra_list = list(solar_like_id_table['Archival_ra'])
dec_list = list(solar_like_id_table['Archival_dec'])
t = Table({'RA': ra_list, 'Dec': dec_list})
aladin.add_table(t, color='red', marker_size=10)